In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

In [ ]:
df = pd.read_csv("data/fraudTrain.csv")

In [ ]:
print('Rows:', len(df))
print(df.isna().sum())
print(df.describe())

In [ ]:
class_counts = df['is_fraud'].value_counts().sort_index()
print(class_counts.rename(index={0: 'Legitimate', 1: 'Fraud'}))
print('Fraud rate: %.3f%%' % (100 * df['is_fraud'].mean()))
class_counts.plot(kind='bar', color=['steelblue', 'tomato'], figsize=(6, 4), title='Class distribution')
plt.xlabel('Transaction class')
plt.ylabel('Count')
plt.xticks([0, 1], ['Legitimate', 'Fraud'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df['distance_km'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
df['dob'] = pd.to_datetime(df['dob'])
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
df['hour'] = df['trans_date_trans_time'].dt.hour

In [ ]:
corr_df = df[['amt', 'age', 'distance_km', 'hour', 'is_fraud']].corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr_df, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_df.columns)), corr_df.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr_df.index)), corr_df.index)
for row in range(len(corr_df.index)):
    for col in range(len(corr_df.columns)):
        ax.text(col, row, f'{corr_df.iloc[row, col]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
features = ['category', 'amt', 'gender', 'age', 'distance_km', 'hour']
X = df[features].copy()
y = df['is_fraud']
X['gender'] = X['gender'].map({'M': 0, 'F': 1})
le_cat = LabelEncoder()
X['category'] = le_cat.fit_transform(X['category'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
smote_preview = SMOTENC(categorical_features=[0, 2], sampling_strategy=0.25,
                        k_neighbors=3, random_state=42)
_, y_train_resampled = smote_preview.fit_resample(X_train, y_train)
balance = pd.DataFrame({
    'Original train': y_train.value_counts(),
    'After SMOTENC': y_train_resampled.value_counts(),
}).fillna(0).astype(int)
display(balance.rename(index={0: 'Legitimate', 1: 'Fraud'}))
balance.rename(index={0: 'Legitimate', 1: 'Fraud'}).plot(kind='bar', figsize=(7, 4), title='Training balance before and after SMOTENC')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
model = Pipeline(steps=[
    ('smote', SMOTENC(categorical_features=[0, 2], sampling_strategy=0.25,
                    k_neighbors=3, random_state=42)),
    ('classifier', RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=1
))
])
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
train_prob = model.predict_proba(X_train)[:, 1]
test_prob = model.predict_proba(X_test)[:, 1]
print("Train ROC-AUC:", roc_auc_score(y_train, train_prob))
print("Test ROC-AUC:", roc_auc_score(y_test, test_prob))
print("Train PR-AUC:", average_precision_score(y_train, train_prob))
print("Test PR-AUC:", average_precision_score(y_test, test_prob))
print("ROC-AUC gap (train - test):", roc_auc_score(y_train, train_prob) - roc_auc_score(y_test, test_prob))
print("PR-AUC gap (train - test):", average_precision_score(y_train, train_prob) - average_precision_score(y_test, test_prob))

In [ ]:
joblib.dump(model, 'model/fraud_model.pkl')
joblib.dump(le_cat, 'model/category_encoder.pkl')
joblib.dump(features, 'model/features.pkl')